# 02 – Preprocessing & Data Cleaning

### Purpose of the Notebook
This notebook applies systematic cleaning and standardisation to the pre‑saved datasets (dataset.pkl and dataset_de.pkl).
All decisions are based on the insights from Notebook 01_data_overview (EDA), including handling of missing data, removal of low‑quality fields, type corrections, logical consistency checks, and creation of derived features.

### Steps
- Load pre‑saved datasets
- Apply preprocessing pypline
- Save cleaned datasets

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.preprocessing import preprocess
from my_scripts.eda import overview

In [5]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset.pkl")

print("EU dataset:", df.shape)

EU dataset: (4039906, 75)


--------------
### Apply preprocessing pipeline
-----------

In [6]:
# apply funktion
df_clean = preprocess(df)


In [7]:
# shape of the cleaned dataset
print(df_clean.shape)

(4039906, 28)


In [8]:
# inspekt of the cleaned dataset
overview(df_clean)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int16,4039906,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int8,4039906,0,0.00,7,"[3, 6, 18, 25, 21, 23, 22]"
DT_DISPATCH,datetime64[us],4039906,0,0.00,3296,"[2007-12-11 00:00:00, 2007-12-27 00:00:00, 200..."
CANCELLED,int8,4039906,0,0.00,2,"[0, 1]"
CORRECTIONS,int8,4039906,0,0.00,10,"[0, 1, 2, 3, 5, 4, 84, 7, 8, 6]"
ISO_COUNTRY_CODE,str,4039906,0,0.00,33,"[DE, FR, ES, SE, PL, IT, HU, CY, UK, RO, PT, N..."
CAE_TYPE,str,4039906,0,0.00,10,"[8, 3, 1, 6, R, N, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,str,4039906,0,0.00,3,"[Unknown, Y, N]"
TYPE_OF_CONTRACT,str,4039906,0,0.00,3,"[W, U, S]"
B_DYN_PURCH_SYST,str,4039906,0,0.00,3,"[Unknown, Y, N]"


In [9]:
# drop irrelevante for target
df_clean = df_clean[df_clean["NUMBER_OFFERS"].notna()].reset_index(drop=True)

In [11]:
df_clean.shape

(3521623, 28)

#### Notes: Summary of Data Cleaning Results

1. Dataset Size After Preprocessing
- Before: 4,039,906 rows × 75 columns
- After: 3,521,623 rows × 28 columns
The reduction from 75 to 28 columns results from removing non‑informative fields, consolidating text columns, and retaining only variables relevant for competition and failure‑risk modelling.
Rows with missing NUMBER_OFFERS were removed because the number of bids is essential for defining the target variable and cannot be reconstructed from any other fields.

2. Columns Removed During Preprocessing
- Columns removed due to >40% missing values (with the exception of CRIT_PRICE_WEIGHT, which was retained due to analytical importance)
  - Winner information (WIN_*)
  - Contracting authority details (CAE_*)
  - GPA-related fields
  - Secondary financial fields (VALUE_EURO_FIN_*, AWARD_VALUE_EURO_FIN_1)
  - Award criteria weights (CRIT_*, except CRIT_PRICE_WEIGHT)
  - Additional CPVs
  - High-cardinality procedural flags (B_MULTIPLE_*, B_FRA_*, FRA_ESTIMATED, etc.)
  - TED_NOTICE_URL

- Columns removed due to irrelevance for competition modelling
  - Identifiers (ID_NOTICE_CAN, ID_AWARD, ID_LOT_AWARDED, CONTRACT_NUMBER)
  - Non-award information (INFO_ON_NON_AWARD, INFO_UNPUBLISHED)
  - Administrative metadata (MAIN_ACTIVITY, EU_INST_CODE)

- Columns removed due to consolidation into a single NLP field
  - TITLE
  - CRIT_CRITERIA
  - CRIT_WEIGHTS
TEXT_ALL is used for all NLP feature extraction steps (TF‑IDF, SVD, NMF, SVM).

3. Key Variables Retained Despite Missing Values, should be replaced by the median
- VALUE_EURO
  - Missing: 39.85% (EU dataset)
  - Importance: baseline contract value, used for log-transformations and value bins.

- AWARD_VALUE_EURO
  - Missing: 27.56% (EU dataset)
  - Importance: actual awarded value, essential for understanding tender dynamics.

- CRIT_PRICE_WEIGHT
  - Missing: 60.85% (EU dataset)
  - Importance: capture how a tender is evaluated—its balance of price vs. quality, the complexity of requirements, and the structure of scoring—making them essential indicators of competitiveness and failure risk.
They are central to the analytical goal (competition modelling). Missingness is informative, not random.

4. Additional Preprocessing Steps
- All categorical missing values were replaced with "Unknown".
- Numeric missing values were left as NaN, to be handled during modelling.

5. Conversion of Categorical Variables
To ensure correct feature engineering and avoid dtype-related errors, all categorical variables were explicitly converted to string (object).
This includes fields such as:
  - CAE_TYPE
  - TYPE_OF_CONTRACT
  - B_EU_FUNDS
  - TOP_TYPE
  - CRIT_CODE
  - B_ELECTRONIC_AUCTION
  - B_AWARDED_TO_A_GROUP
  - B_CONTRACTOR_SME
  - B_SUBCONTRACTED
This guarantees correct handling of categorical data and enables creation of CPV hierarchy features in the feature engineering stage.

6. High‑cardinality categorical features were removed because they contain hundreds or thousands of unique values, cannot be meaningfully encoded,and cause memory issues during preprocessing.
- Removed columns:
  - TAL_LOCATION_NUTS (9310 uniques)
  - ID_LOT (3323 uniques)
  - WIN_COUNTRY_CODE (708 uniques)
  - CPV_GROUP (470 uniques)
  - CPV_CLASS (1836 uniques)
  - XSD_VERSION (technical metadata, no modelling value)

- CPV_DIVISION was aggregated into 12 business‑interpretable categories to reduce dimensionality and provide a clear, user‑friendly input for the simulator. New column: CPV_CATEGORY (12 groups)

- Integer columns were downcast (int64 into int8/int16) to reduce memory usage and improve preprocessing performance.
- Affected columns: all numeric columns with small value ranges (binary flags, counters, IDs with limited range)

--------------
### Save cleaned datasets

-----------

In [12]:
df_clean.to_pickle("../data/dataset_clean.pkl")